# Generate forest plots 

In [2]:
## Import the necessary packages 
import os
import numpy as np
import pandas as pd
import math
import sys
import subprocess
from scipy import stats
import statsmodels.api as sm
import scipy
from scipy import stats
from scipy.stats import chi2
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

## Print out package versions
## Getting packages loaded into this notebook and their versions to allow for reproducibility
import pkg_resources
import types
from datetime import date

today = date.today()
date = today.strftime("%d-%b-%Y").upper()

## Define function 
def get_imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            name = val.__name__.split(".")[0]
        elif isinstance(val, type):
            name = val.__module__.split(".")[0]

        poorly_named_packages = {
            "PIL": "Pillow",
            "sklearn": "scikit-learn"
        }
        if name in poorly_named_packages:
            name = poorly_named_packages[name]

        yield name

## Get a list of packages imported 
imports = list(set(get_imports()))

requirements = []
for m in pkg_resources.working_set:
    if m.project_name in imports and m.project_name != "pip":
        requirements.append((m.project_name, m.version))

## Print out packages and versions 
print(f"PACKAGE VERSIONS ({date})")
for r in requirements:
    print("\t{}=={}".format(*r))

## Also print which Python is being used
print("\nPYTHON INFO")
print(f"\tPython executable: {sys.executable}")

/tmp/ipykernel_1792494/3049236665.py:19: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


PACKAGE VERSIONS (06-DEC-2025)
	matplotlib==3.8.4
	numpy==1.26.4
	pandas==2.2.3
	scipy==1.13.1
	seaborn==0.13.2
	statsmodels==0.14.4

PYTHON INFO
	Python executable: /usr/local/apps/python/py3.11/bin/python


## Calculate 

In [3]:
st2 = pd.read_csv(f"{DATA_DIR}/forest_plot/all_SNPs_by_cohort.txt", sep="\t")

In [4]:
def calculate_or_ci(beta, se, confidence_level=0.95):
    # Calculate Z-score for confidence level
    z = np.abs(stats.norm.ppf((1 - confidence_level) / 2))
    
    # Calculate OR and confidence intervals
    OR = np.exp(beta)
    L95 = np.exp(beta - z * se)
    U95 = np.exp(beta + z * se)
    
    return OR, L95, U95

# For a DataFrame:
def add_or_ci_to_dataframe(df, beta_col='BETA', se_col='SE', 
                           or_col='OR', l95_col='L95', u95_col='U95'):
    df = df.copy()
    
    # Calculate OR and CI
    df[or_col], df[l95_col], df[u95_col] = calculate_or_ci(
        df[beta_col].values, 
        df[se_col].values
    )
    return df


In [5]:
st2['BETA'] = pd.to_numeric(st2['BETA'], errors='coerce')
st2['SE'] = pd.to_numeric(st2['SE'], errors='coerce')

st2_with_or = add_or_ci_to_dataframe(st2)

In [6]:
st2_with_or.to_csv(f"{DATA_DIR}/forest_plot/ST2_complete.txt", sep="\t")

In [7]:
st2_with_or.head(20)

,rsID,SNP(hg38),Cohort,BETA,SE,P,OR,L95,U95
0,rs3115534,chr1:155235878:G:T,[AFR] GP2,-0.54839,0.04276,1.20E-37,0.577879,0.531422,0.628398
1,rs3115534,chr1:155235878:G:T,[AAC] GP2,-0.77191,0.12819,1.73E-09,0.462130,0.359458,0.594127
2,rs3115534,chr1:155235878:G:T,[AAC] 23andMe,-0.35507,0.11114,2.03E-03,0.701124,0.563888,0.871761
3,rs3115534,chr1:155235878:G:T,[AAC] MVP,-0.18232,0.07215,1.15E-02,0.833335,0.723445,0.959917
4,rs3115534,chr1:155235878:G:T,[AAC] Meta-analysis,-0.33170,0.05470,1.36E-09,0.717703,0.644739,0.798924
5,rs3115534,chr1:155235878:G:T,[AFR/AAC] Combined Meta-analysis,-0.46620,0.03370,1.53E-43,0.627382,0.587282,0.670220
6,rs11547135,chr4:76213633:C:T,[AFR] GP2,0.23506,0.04513,1.91E-07,1.264985,1.157899,1.381974
7,rs11547135,chr4:76213633:C:T,[AAC] GP2,0.20861,0.10111,3.91E-02,1.231964,1.010493,1.501977
8,rs11547135,chr4:76213633:C:T,[AAC] 23andMe,0.09995,0.09815,3.12E-01,1.105116,0.911722,1.339532
9,rs11547135,chr4:76213633:C:T,[AAC] MVP,0.09266,0.06687,1.66E-01,1.097089,0.962325,1.250724


## Add EAF

## Plots

In [ ]:
# Define cohort order
order = [
    "[AFR] GP2",
    "[AAC] GP2",
    "[AAC] 23andMe",
    "[AAC] MVP",
    "[AAC] Meta-analysis",
    "[AFR/AAC] Combined Meta-analysis",
]

# Get unique rsIDs
unique_rsids = st2_with_or['rsID'].unique()

# Loop through each variant
for rsid in unique_rsids:
    # Filter data for this variant
    df = st2_with_or[st2_with_or['rsID'] == rsid].copy()
    
    # Skip if no valid data
    if df['OR'].isna().all():
        print(f"Skipping {rsid} - no valid OR data")
        continue
    
    # Clean and order cohorts
    df["Cohort"] = df["Cohort"].astype(str).str.strip()
    df["Cohort"] = pd.Categorical(df["Cohort"], categories=order, ordered=True)
    df = df.sort_values("Cohort").reset_index(drop=True)
    
    # Remove rows with missing OR/CI values
    df = df.dropna(subset=['OR', 'L95', 'U95'])
    
    # Convert P-value to float if it's not already
    df['P'] = pd.to_numeric(df['P'], errors='coerce')
    
    if len(df) == 0:
        print(f"Skipping {rsid} - no valid data after filtering")
        continue
    
    # Y positions
    y = np.arange(len(df))
    
    # Autoscale x with padding
    xmin = float(df["L95"].min()) * 0.9
    xmax = float(df["U95"].max()) * 1.1
    
    # Create plot
    fig, ax = plt.subplots(figsize=(10, max(6, len(df) * 0.6)))
    ax.set_xlim(xmin, xmax)
    
    # Plot CIs and point estimates
    ax.errorbar(
        df["OR"], y,
        xerr=[df["OR"] - df["L95"], df["U95"] - df["OR"]],
        fmt="s", color="black", ecolor="black", elinewidth=1, capsize=3
    )
    
    # Reference line at OR = 1
    ax.axvline(1, color="grey", linestyle="--", linewidth=1)
    
    # Y labels
    ax.set_yticks(y)
    ax.set_yticklabels(df["Cohort"])
    ax.invert_yaxis()
    
    # Right-side text block
    right = ax.get_xlim()[1]
    x_text = right * 1.05
    
    # Add row values
    for yi, (OR, L95, U95, p) in enumerate(zip(df["OR"], df["L95"], df["U95"], df["P"])):
        label = f"OR:{OR:.2f} [{L95:.2f}, {U95:.2f}]; P={p:.2E}"
        if p < 5e-8:
            label += " *"
        ax.text(x_text, yi, label, va="center", ha="left", fontsize=9, color="black")
    
    # Labels and title
    ax.set_xlabel("Odds Ratio (95% CI)")
    
    # Get SNP and EA info for title
    snp = df['SNP(hg38)'].iloc[0]
    ea = df['EA'].iloc[0]
    ax.set_title(f"SNP: {snp} | rsID: {rsid}")
    
    plt.tight_layout()
    
    # Replace ":" with "_" in filenames for safety
    safe_snp = snp.replace(":", "_")
    outfile = os.path.join(f"{DATA_DIR}/forest_plot/plots", f"{safe_snp}_{rsid}_forest.png")
    
    # Save with rsID in filename
    plt.savefig(outfile, dpi=300, bbox_inches="tight")
    print(f"Saved {outfile}")
    
    plt.show()
    plt.close()